# Infaction — Site grouping with Farm Family and Ship Family
This companion runs the existing executable v0.4.6 world→placement→city baseline, followed by original Book1 **Proximity** Site grouping and the **targeted Farm/Ship family post-filter v2**.

**Directional Containment is canceled.** Farm/Dock/Shipyard Site and Farm/Ship/Dock Location relationships are filtered only after greedy Site generation; unrelated types keep original Proximity semantics. Canceled/removed memberships are returned to their original Zone or standalone City pool.

**Runtime status:** executable historical selection pipeline; latest v0.4.7 71-Location eligibility and function-minimum repair are not yet migrated into this companion. Do not label its selection output as fully integrated v0.4.7. Original Book1 and historical notebooks are untouched.


In [ ]:
import urllib.request,json,contextlib,io,types
ROOT="https://raw.githubusercontent.com/playrenathea/rantau/main/"
def get_text(path):return urllib.request.urlopen(ROOT+path,timeout=60).read().decode("utf-8")
base=json.loads(get_text("notebooks/Infaction_Locked_City_Placement_v0.4.6.ipynb"))
scope={"__name__":"__main__"}
for idx in (1,2,3):
    exec(compile("".join(base["cells"][idx]["source"]),f"base_cell_{idx}","exec"),scope)
print("Canonical old executable selection/placement/City preflight passed")


In [ ]:
from pathlib import Path
site_mod=types.ModuleType("infaction_site_proximity_checkpoint")
exec(compile(get_text("infaction_site_proximity_checkpoint.py"),"infaction_site_proximity_checkpoint.py","exec"),site_mod.__dict__)
master=scope["MASTER"]
scales={name:int(row["scale"]) for name,row in master["raw"]["place_catalog"].items()}
effective={**scales,**{"Fortress":3,"Palace":3,"Mine":3,"Factory":3,"Town Hall":2}}
site_candidates=[p for p in scope["place_names"] if effective.get(p) in (2,3,4)]
assert len(site_candidates)==50
site_result=site_mod.group_sites(scope["placed"],scope["result"],scope["SITE_AFFINITY"],site_candidates,scope["city_scores"],seed=scope["SEED"])
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
family_mod=types.ModuleType("infaction_targeted_farm_ship_site_prune_v2")
exec(compile(get_text("experiments/infaction_targeted_farm_ship_site_prune_v2.py"),"infaction_targeted_farm_ship_site_prune_v2.py","exec"),family_mod.__dict__)
family_rules=json.loads(get_text("experiments/infaction_targeted_farm_ship_site_filter_v2.json"))
site_records=[{"site":s["name"],"container":("z:"+s["parent_zone"] if s["parent_zone"] is not None else s["city"].lower()),"children":s["children"]} for s in site_result["sites"]]
family_result=family_mod.prune_sites(site_records,family_rules,proximity=scope["SITE_AFFINITY"])
sites_by_name={s["name"]:s for s in site_result["sites"]}
retained_names={r["site"] for r in family_result["sites"]}
retained_sites=[{**sites_by_name[r["site"]],"children":r["children"]} for r in family_result["sites"]]
remaining_by_zone={z:list(members) for z,members in site_result["direct_locations_by_zone"].items()}
remaining_inside=list(site_result["standalone_ungrouped_inside"])
remaining_outside=list(site_result["standalone_ungrouped_outside"])
for container,children in family_result["direct_restored"].items():
    if container.startswith("z:"):
        remaining_by_zone[container[2:]].extend(children)
    elif container=="inside":
        remaining_inside.extend(children)
    elif container=="outside":
        remaining_outside.extend(children)
    else:
        raise AssertionError("Unknown restored container")
members=[loc for s in retained_sites for loc in s["children"]]
members+=sum(remaining_by_zone.values(),[])+remaining_inside+remaining_outside
assert len(members)==len(set(members))==len(scope["placed"]["locations"])
assert set(members)==set(scope["placed"]["locations"])
site_result["sites"]=retained_sites
site_result["direct_locations_by_zone"]=remaining_by_zone
site_result["standalone_ungrouped_inside"]=remaining_inside
site_result["standalone_ungrouped_outside"]=remaining_outside
site_result["grouped_location_count"]=sum(len(s["children"]) for s in retained_sites)
site_result["ungrouped_location_count"]=len(members)-site_result["grouped_location_count"]
site_result["family_filter"]={k:v for k,v in family_result.items() if k!="sites"}
site_result["status"]="Proximity with Farm/Ship family post-filter v2; historical selection executable"
print("Seed",scope["SEED"],"Sites",len(site_result["sites"]),"grouped",site_result["grouped_location_count"],"canceled",len(family_result["canceled"]))
for s in site_result["sites"]:print(s["name"],"in",s["parent_zone"] or "Standalone",":",", ".join(s["children"]))
Path(f'infaction_world_with_sites_experimental_{scope["SEED"]}.json').write_text(json.dumps({"world":scope["world"],"placement":scope["placed"],"city":scope["result"],"site_grouping":site_result},indent=2,ensure_ascii=False))
